In [132]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [133]:
PROJECT_ROOT = Path("..").resolve()

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "observations.parquet"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "phase2_labeled_observations.parquet"
)

REPORT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "anomaly_injection_report.json"
)

print("Input exists:", INPUT_FILE.exists())
print(INPUT_FILE)

Input exists: True
C:\Anomaly_Detection\data\processed\observations.parquet


In [134]:
df = pd.read_parquet(INPUT_FILE)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Rows: 420551
Columns: ['timestamp', 'station_id', 'temperature_c', 'pressure_hpa', 'relative_humidity_pct']


,timestamp,station_id,temperature_c,pressure_hpa,relative_humidity_pct
0,2009-01-01 00:10:00,JENA_001,-8.02,996.52,93.3
1,2009-01-01 00:20:00,JENA_001,-8.41,996.57,93.4
2,2009-01-01 00:30:00,JENA_001,-8.51,996.53,93.9
3,2009-01-01 00:40:00,JENA_001,-8.31,996.51,94.2
4,2009-01-01 00:50:00,JENA_001,-8.27,996.51,94.1


In [135]:
required_columns = [
    "timestamp",
    "station_id",
    "temperature_c",
    "pressure_hpa",
    "relative_humidity_pct"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing Phase 1 columns: {missing_columns}"
    )

assert len(df) > 0
assert df["timestamp"].notna().all()

print("Phase 1 input validation passed.")

Phase 1 input validation passed.


In [136]:
phase2_df = df.drop_duplicates().copy()

phase2_df = (
    phase2_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Phase 1 rows :", len(df))
print("Phase 2 rows :", len(phase2_df))
print("Removed exact duplicates:", len(df) - len(phase2_df))

Phase 1 rows : 420551
Phase 2 rows : 420224
Removed exact duplicates: 327


In [137]:
duplicate_timestamps = phase2_df.duplicated(
    subset=["station_id", "timestamp"]
).sum()

print("Remaining duplicate timestamps:", duplicate_timestamps)

Remaining duplicate timestamps: 0


In [138]:
phase2_df["original_temperature_c"] = phase2_df["temperature_c"]
phase2_df["original_pressure_hpa"] = phase2_df["pressure_hpa"]
phase2_df["original_relative_humidity_pct"] = (
    phase2_df["relative_humidity_pct"]
)

phase2_df.head()

,timestamp,station_id,temperature_c,pressure_hpa,relative_humidity_pct,original_temperature_c,original_pressure_hpa,original_relative_humidity_pct
0,2009-01-01 00:10:00,JENA_001,-8.02,996.52,93.3,-8.02,996.52,93.3
1,2009-01-01 00:20:00,JENA_001,-8.41,996.57,93.4,-8.41,996.57,93.4
2,2009-01-01 00:30:00,JENA_001,-8.51,996.53,93.9,-8.51,996.53,93.9
3,2009-01-01 00:40:00,JENA_001,-8.31,996.51,94.2,-8.31,996.51,94.2
4,2009-01-01 00:50:00,JENA_001,-8.27,996.51,94.1,-8.27,996.51,94.1


In [139]:
phase2_df["is_anomaly"] = 0
phase2_df["anomaly_type"] = "normal"
phase2_df["anomaly_parameter"] = "none"
phase2_df["anomaly_magnitude"] = 0.0

phase2_df.head()

,timestamp,station_id,temperature_c,pressure_hpa,relative_humidity_pct,original_temperature_c,original_pressure_hpa,original_relative_humidity_pct,is_anomaly,anomaly_type,anomaly_parameter,anomaly_magnitude
0,2009-01-01 00:10:00,JENA_001,-8.02,996.52,93.3,-8.02,996.52,93.3,0,normal,none,0.0
1,2009-01-01 00:20:00,JENA_001,-8.41,996.57,93.4,-8.41,996.57,93.4,0,normal,none,0.0
2,2009-01-01 00:30:00,JENA_001,-8.51,996.53,93.9,-8.51,996.53,93.9,0,normal,none,0.0
3,2009-01-01 00:40:00,JENA_001,-8.31,996.51,94.2,-8.31,996.51,94.2,0,normal,none,0.0
4,2009-01-01 00:50:00,JENA_001,-8.27,996.51,94.1,-8.27,996.51,94.1,0,normal,none,0.0


In [140]:
n = len(phase2_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

phase2_df["split"] = "test"

phase2_df.loc[:train_end - 1, "split"] = "train"
phase2_df.loc[train_end:val_end - 1, "split"] = "validation"

print(
    phase2_df["split"].value_counts()
)

split
train         294156
validation     63034
test           63034
Name: count, dtype: int64


In [141]:
for split_name in ["train", "validation", "test"]:
    part = phase2_df[
        phase2_df["split"] == split_name
    ]

    print(
        split_name,
        part["timestamp"].min(),
        "→",
        part["timestamp"].max(),
        len(part)
    )

train 2009-01-01 00:10:00 → 2014-08-05 18:40:00 294156
validation 2014-08-05 18:50:00 → 2015-10-18 04:10:00 63034
test 2015-10-18 04:20:00 → 2017-01-01 00:00:00 63034


In [142]:
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

print("Random seed:", RANDOM_SEED)

Random seed: 42


In [143]:
sensor_columns = {
    "temperature": "temperature_c",
    "pressure": "pressure_hpa",
    "humidity": "relative_humidity_pct"
}

sensor_columns

{'temperature': 'temperature_c',
 'pressure': 'pressure_hpa',
 'humidity': 'relative_humidity_pct'}

In [144]:
train_df = phase2_df[
    phase2_df["split"] == "train"
].copy()

robust_stats = {}

for name, column in sensor_columns.items():

    q1 = train_df[column].quantile(0.25)
    q3 = train_df[column].quantile(0.75)
    iqr = q3 - q1

    median = train_df[column].median()
    std = train_df[column].std()

    robust_stats[name] = {
        "median": float(median),
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "std": float(std)
    }

robust_stats

{'temperature': {'median': 9.18,
  'q1': 2.98,
  'q3': 15.32,
  'iqr': 12.34,
  'std': 8.651037990442015},
 'pressure': {'median': 989.11,
  'q1': 983.61,
  'q3': 994.09,
  'iqr': 10.480000000000018,
  'std': 8.299070694383207},
 'humidity': {'median': 79.2,
  'q1': 65.05,
  'q3': 89.4,
  'iqr': 24.35000000000001,
  'std': 16.53492949441991}}

In [145]:
SPIKE_MULTIPLIERS = {
    "temperature": (0.8, 2.0),
    "pressure": (0.8, 2.0),
    "humidity": (0.5, 1.2)
}

BIAS_MULTIPLIERS = {
    "temperature": (0.3, 0.8),
    "pressure": (0.3, 0.8),
    "humidity": (0.2, 0.6)
}

DRIFT_MULTIPLIERS = {
    "temperature": (0.4, 1.2),
    "pressure": (0.4, 1.2),
    "humidity": (0.3, 0.9)
}

MULTIVARIATE_MULTIPLIERS = {
    "temperature": (0.4, 1.0),
    "pressure": (0.4, 1.0),
    "humidity": (0.3, 0.8)
}

In [146]:
TARGET_ANOMALY_RATE = 0.03

split_budgets = {}

for split_name in [
    "train",
    "validation",
    "test"
]:
    split_size = int(
        (phase2_df["split"] == split_name).sum()
    )

    split_budgets[split_name] = int(
        split_size * TARGET_ANOMALY_RATE
    )

print("Target anomaly rate per split:", TARGET_ANOMALY_RATE)
print("Split anomaly budgets:", split_budgets)
print(
    "Total target anomalies:",
    sum(split_budgets.values())
)

Target anomaly rate per split: 0.03
Split anomaly budgets: {'train': 8824, 'validation': 1891, 'test': 1891}
Total target anomalies: 12606


In [147]:
anomaly_distribution = {
    "spike": 0.25,
    "drift": 0.20,
    "freeze": 0.20,
    "bias": 0.15,
    "noise": 0.10,
    "dropout": 0.05,
    "multivariate_inconsistency": 0.05
}

assert abs(
    sum(anomaly_distribution.values()) - 1.0
) < 1e-9

anomaly_distribution

{'spike': 0.25,
 'drift': 0.2,
 'freeze': 0.2,
 'bias': 0.15,
 'noise': 0.1,
 'dropout': 0.05,
 'multivariate_inconsistency': 0.05}

In [148]:
split_type_budgets = {}

for split_name, total_budget in split_budgets.items():

    budgets = {}
    allocated = 0

    anomaly_types = list(
        anomaly_distribution.keys()
    )

    for anomaly_type in anomaly_types[:-1]:

        count = int(
            total_budget
            * anomaly_distribution[anomaly_type]
        )

        budgets[anomaly_type] = count
        allocated += count

    # Give rounding remainder to final type
    last_type = anomaly_types[-1]

    budgets[last_type] = (
        total_budget - allocated
    )

    split_type_budgets[split_name] = budgets

split_type_budgetssplit_type_budgets = {}

for split_name, total_budget in split_budgets.items():

    budgets = {}
    allocated = 0

    anomaly_types = list(
        anomaly_distribution.keys()
    )

    for anomaly_type in anomaly_types[:-1]:

        count = int(
            total_budget
            * anomaly_distribution[anomaly_type]
        )

        budgets[anomaly_type] = count
        allocated += count

    # Give rounding remainder to final type
    last_type = anomaly_types[-1]

    budgets[last_type] = (
        total_budget - allocated
    )

    split_type_budgets[split_name] = budgets

split_type_budgets

{'train': {'spike': 2206,
  'drift': 1764,
  'freeze': 1764,
  'bias': 1323,
  'noise': 882,
  'dropout': 441,
  'multivariate_inconsistency': 444},
 'validation': {'spike': 472,
  'drift': 378,
  'freeze': 378,
  'bias': 283,
  'noise': 189,
  'dropout': 94,
  'multivariate_inconsistency': 97},
 'test': {'spike': 472,
  'drift': 378,
  'freeze': 378,
  'bias': 283,
  'noise': 189,
  'dropout': 94,
  'multivariate_inconsistency': 97}}

In [149]:
def get_clean_indices(
    dataframe,
    count,
    split_name
):
    candidates = dataframe.index[
        (dataframe["split"] == split_name)
        &
        (dataframe["is_anomaly"] == 0)
    ].to_numpy()

    if count > len(candidates):
        raise ValueError(
            f"Not enough clean rows in {split_name}"
        )

    return rng.choice(
        candidates,
        size=count,
        replace=False
    )

In [150]:
def inject_spikes(
    dataframe,
    count,
    split_name
):
    indices = get_clean_indices(
        dataframe,
        count,
        split_name
    )

    parameters = rng.choice(
        list(sensor_columns.keys()),
        size=count
    )

    for idx, parameter in zip(
        indices,
        parameters
    ):
        column = sensor_columns[parameter]

        iqr = robust_stats[parameter]["iqr"]

        direction = rng.choice([-1, 1])

        low, high = SPIKE_MULTIPLIERS[parameter]
        factor = rng.uniform(low, high)

        magnitude = direction * factor * iqr

        dataframe.at[idx, column] += magnitude

        dataframe.at[idx, "is_anomaly"] = 1
        dataframe.at[idx, "anomaly_type"] = "spike"
        dataframe.at[idx, "anomaly_parameter"] = parameter
        dataframe.at[idx, "anomaly_magnitude"] = float(magnitude)

In [151]:
print("Spike injection function ready.")

Spike injection function ready.


In [152]:
def inject_bias(
    dataframe,
    target_rows,
    split_name,
    min_window=6,
    max_window=36
):
    injected = 0

    split_indices = dataframe.index[
        dataframe["split"] == split_name
    ].to_numpy()

    while injected < target_rows:

        start = int(
            rng.choice(split_indices)
        )

        window_size = int(
            rng.integers(
                min_window,
                max_window + 1
            )
        )

        end = min(
            start + window_size,
            len(dataframe)
        )

        window_indices = dataframe.index[
            (dataframe.index >= start)
            &
            (dataframe.index < end)
            &
            (dataframe["split"] == split_name)
            &
            (dataframe["is_anomaly"] == 0)
        ]

        if len(window_indices) == 0:
            continue

        parameter = rng.choice(
            list(sensor_columns.keys())
        )

        column = sensor_columns[parameter]

        iqr = robust_stats[parameter]["iqr"]

        direction = rng.choice([-1, 1])

        low, high = BIAS_MULTIPLIERS[parameter]

        magnitude = float(
            direction
            * rng.uniform(low, high)
            * iqr
        )

        remaining = (
            target_rows - injected
        )

        selected_indices = (
            window_indices[:remaining]
        )

        dataframe.loc[
            selected_indices,
            column
        ] += magnitude

        dataframe.loc[
            selected_indices,
            "is_anomaly"
        ] = 1

        dataframe.loc[
            selected_indices,
            "anomaly_type"
        ] = "bias"

        dataframe.loc[
            selected_indices,
            "anomaly_parameter"
        ] = parameter

        dataframe.loc[
            selected_indices,
            "anomaly_magnitude"
        ] = magnitude

        injected += len(
            selected_indices
        )

In [153]:
print("Bias injection function ready.")

Bias injection function ready.


In [154]:
def inject_drift(
    dataframe,
    target_rows,
    split_name,
    min_window=12,
    max_window=72
):
    injected = 0

    split_indices = dataframe.index[
        dataframe["split"] == split_name
    ].to_numpy()
    while injected < target_rows:

        start = int(
            rng.choice(split_indices)
        )

        window_size = int(
            rng.integers(
                min_window,
                max_window + 1
            )
        )

        end = min(
            start + window_size,
            len(dataframe)
        )

        window_indices = dataframe.index[
            (dataframe.index >= start)
            &
            (dataframe.index < end)
            &
            (dataframe["split"] == split_name)
            &
            (dataframe["is_anomaly"] == 0)
        ]

        if len(window_indices) < 3:
            continue

        remaining = (
            target_rows - injected
        )

        selected_indices = (
            window_indices[:remaining]
        )

        parameter = rng.choice(
            list(sensor_columns.keys())
        )

        column = sensor_columns[parameter]

        iqr = robust_stats[parameter]["iqr"]

        direction = rng.choice([-1, 1])

        low, high = DRIFT_MULTIPLIERS[parameter]

        max_drift = float(
            direction
            * rng.uniform(low, high)
            * iqr
        )

        drift_values = np.linspace(
            0,
            max_drift,
            len(selected_indices)
        )

        dataframe.loc[
            selected_indices,
            column
        ] += drift_values

        dataframe.loc[
            selected_indices,
            "is_anomaly"
        ] = 1

        dataframe.loc[
            selected_indices,
            "anomaly_type"
        ] = "drift"

        dataframe.loc[
            selected_indices,
            "anomaly_parameter"
        ] = parameter

        dataframe.loc[
            selected_indices,
            "anomaly_magnitude"
        ] = drift_values

        injected += len(
            selected_indices
        )

In [155]:
print("Drift injection function ready.")

Drift injection function ready.


In [156]:
def inject_freeze(
    dataframe,
    target_rows,
    split_name,
    min_window=6,
    max_window=36
):
    injected = 0

    split_indices = dataframe.index[
        dataframe["split"] == split_name
    ].to_numpy()

    while injected < target_rows:

        start = int(
            rng.choice(split_indices)
        )

        window_size = int(
            rng.integers(
                min_window,
                max_window + 1
            )
        )

        end = min(
            start + window_size,
            len(dataframe)
        )

        window_indices = dataframe.index[
            (dataframe.index >= start)
            &
            (dataframe.index < end)
            &
            (dataframe["split"] == split_name)
            &
            (dataframe["is_anomaly"] == 0)
        ]

        if len(window_indices) < 3:
            continue

        remaining = (
            target_rows - injected
        )

        selected_indices = (
            window_indices[:remaining]
        )

        parameter = rng.choice(
            list(sensor_columns.keys())
        )

        column = sensor_columns[parameter]

        freeze_value = dataframe.at[
            selected_indices[0],
            column
        ]

        original_values = dataframe.loc[
            selected_indices,
            column
        ].copy()

        dataframe.loc[
            selected_indices,
            column
        ] = freeze_value

        magnitudes = (
            freeze_value
            - original_values
        ).astype(float)

        dataframe.loc[
            selected_indices,
            "is_anomaly"
        ] = 1

        dataframe.loc[
            selected_indices,
            "anomaly_type"
        ] = "freeze"

        dataframe.loc[
            selected_indices,
            "anomaly_parameter"
        ] = parameter

        dataframe.loc[
            selected_indices,
            "anomaly_magnitude"
        ] = magnitudes.values

        injected += len(
            selected_indices
        )

In [157]:
print("Freeze injection function ready.")

Freeze injection function ready.


In [158]:
def inject_noise(
    dataframe,
    count,
    split_name
):
    indices = get_clean_indices(
        dataframe,
        count,
        split_name
    )

    parameters = rng.choice(
        list(sensor_columns.keys()),
        size=count
    )

    for idx, parameter in zip(
        indices,
        parameters
    ):
        column = sensor_columns[parameter]

        std = robust_stats[parameter]["std"]

        noise = float(
            rng.normal(
                loc=0,
                scale=0.8 * std
            )
        )

        dataframe.at[
            idx,
            column
        ] += noise

        dataframe.at[
            idx,
            "is_anomaly"
        ] = 1

        dataframe.at[
            idx,
            "anomaly_type"
        ] = "noise"

        dataframe.at[
            idx,
            "anomaly_parameter"
        ] = parameter

        dataframe.at[
            idx,
            "anomaly_magnitude"
        ] = noise

In [159]:
print("Noise injection function ready.")

Noise injection function ready.


In [160]:
def inject_dropout(
    dataframe,
    count,
    split_name
):
    indices = get_clean_indices(
        dataframe,
        count,
        split_name
    )
    parameters = rng.choice(
        list(sensor_columns.keys()),
        size=count
    )

    for idx, parameter in zip(
        indices,
        parameters
    ):
        column = sensor_columns[parameter]

        original_value = dataframe.at[
            idx,
            column
        ]

        dataframe.at[
            idx,
            column
        ] = np.nan

        dataframe.at[
            idx,
            "is_anomaly"
        ] = 1

        dataframe.at[
            idx,
            "anomaly_type"
        ] = "dropout"

        dataframe.at[
            idx,
            "anomaly_parameter"
        ] = parameter

        dataframe.at[
            idx,
            "anomaly_magnitude"
        ] = float(original_value)

In [161]:
print("Dropout injection function ready.")

Dropout injection function ready.


In [162]:
def inject_multivariate_inconsistency(
    dataframe,
    count,
    split_name
):
    indices = get_clean_indices(
        dataframe,
        count,
        split_name
    )

    parameters = rng.choice(
        list(sensor_columns.keys()),
        size=count
    )

    for idx, parameter in zip(
        indices,
        parameters
    ):
        column = sensor_columns[parameter]

        iqr = robust_stats[parameter]["iqr"]

        direction = rng.choice([-1, 1])

        low, high = MULTIVARIATE_MULTIPLIERS[parameter]

        magnitude = float(
            direction
            * rng.uniform(low, high)
            * iqr
        )

        dataframe.at[
            idx,
            column
        ] += magnitude

        dataframe.at[
            idx,
            "is_anomaly"
        ] = 1

        dataframe.at[
            idx,
            "anomaly_type"
        ] = "multivariate_inconsistency"

        dataframe.at[
            idx,
            "anomaly_parameter"
        ] = parameter

        dataframe.at[
            idx,
            "anomaly_magnitude"
        ] = magnitude

In [163]:
for split_name in [
    "train",
    "validation",
    "test"
]:

    budgets = split_type_budgets[
        split_name
    ]

    inject_spikes(
        phase2_df,
        budgets["spike"],
        split_name
    )

    inject_drift(
        phase2_df,
        budgets["drift"],
        split_name
    )

    inject_freeze(
        phase2_df,
        budgets["freeze"],
        split_name
    )

    inject_bias(
        phase2_df,
        budgets["bias"],
        split_name
    )

    inject_noise(
        phase2_df,
        budgets["noise"],
        split_name
    )

    inject_dropout(
        phase2_df,
        budgets["dropout"],
        split_name
    )

    inject_multivariate_inconsistency(
        phase2_df,
        budgets["multivariate_inconsistency"],
        split_name
    )

print("Anomaly injection complete.")

Anomaly injection complete.


In [164]:
total_anomalies = int(
    phase2_df["is_anomaly"].sum()
)

actual_anomaly_rate = (
    total_anomalies
    / len(phase2_df)
)

print("Total anomalies:", total_anomalies)

print(
    "Overall anomaly rate:",
    f"{actual_anomaly_rate:.4%}"
)

print()

for split_name in [
    "train",
    "validation",
    "test"
]:

    subset = phase2_df[
        phase2_df["split"] == split_name
    ]

    print(
        f"{split_name:<12} "
        f"{int(subset['is_anomaly'].sum()):>6,} "
        f"{subset['is_anomaly'].mean():.4%}"
    )

Total anomalies: 12606
Overall anomaly rate: 2.9998%

train         8,824 2.9998%
validation    1,891 3.0000%
test          1,891 3.0000%


In [165]:
pd.crosstab(
    phase2_df["split"],
    phase2_df["anomaly_type"]
)

anomaly_type,bias,drift,dropout,freeze,multivariate_inconsistency,noise,normal,spike
split,,,,,,,,
test,283,378,94,378,97,189,61143,472
train,1323,1764,441,1764,444,882,285332,2206
validation,283,378,94,378,97,189,61143,472


In [166]:
anomalies_only = phase2_df[
    phase2_df["is_anomaly"] == 1
]

pd.crosstab(
    anomalies_only["split"],
    anomalies_only["anomaly_parameter"]
)

anomaly_parameter,humidity,pressure,temperature
split,,,
test,700,634,557
train,2847,2827,3150
validation,551,626,714


In [167]:
pd.crosstab(
    phase2_df["split"],
    phase2_df["is_anomaly"]
)

is_anomaly,0,1
split,,
test,61143,1891
train,285332,8824
validation,61143,1891


In [168]:
split_summary = (
    phase2_df
    .groupby("split")
    .agg(
        total_rows=("is_anomaly", "size"),
        anomaly_rows=("is_anomaly", "sum")
    )
)

split_summary["normal_rows"] = (
    split_summary["total_rows"]
    - split_summary["anomaly_rows"]
)

split_summary["anomaly_rate"] = (
    split_summary["anomaly_rows"]
    / split_summary["total_rows"]
)

split_summary

,total_rows,anomaly_rows,normal_rows,anomaly_rate
split,,,,
test,63034,1891,61143,0.030000
train,294156,8824,285332,0.029998
validation,63034,1891,61143,0.030000


In [169]:
type_by_split = pd.crosstab(
    phase2_df["split"],
    phase2_df["anomaly_type"]
)

type_by_split

anomaly_type,bias,drift,dropout,freeze,multivariate_inconsistency,noise,normal,spike
split,,,,,,,,
test,283,378,94,378,97,189,61143,472
train,1323,1764,441,1764,444,882,285332,2206
validation,283,378,94,378,97,189,61143,472


In [170]:
anomalies_only = phase2_df[
    phase2_df["is_anomaly"] == 1
]

parameter_by_split = pd.crosstab(
    anomalies_only["split"],
    anomalies_only["anomaly_parameter"]
)

parameter_by_split

anomaly_parameter,humidity,pressure,temperature
split,,,
test,700,634,557
train,2847,2827,3150
validation,551,626,714


In [171]:
assert phase2_df["is_anomaly"].isin([0, 1]).all()

assert (
    phase2_df.loc[
        phase2_df["is_anomaly"] == 0,
        "anomaly_type"
    ] == "normal"
).all()

assert (
    phase2_df.loc[
        phase2_df["is_anomaly"] == 0,
        "anomaly_parameter"
    ] == "none"
).all()

assert (
    phase2_df.loc[
        phase2_df["is_anomaly"] == 1,
        "anomaly_type"
    ] != "normal"
).all()

assert (
    phase2_df.loc[
        phase2_df["is_anomaly"] == 1,
        "anomaly_parameter"
    ] != "none"
).all()

print("Label consistency passed.")

Label consistency passed.


In [172]:
original_columns = [
    "original_temperature_c",
    "original_pressure_hpa",
    "original_relative_humidity_pct"
]

for column in original_columns:
    assert phase2_df[column].notna().all()

print("Original sensor values preserved.")

Original sensor values preserved.


In [173]:
normal_mask = (
    phase2_df["is_anomaly"] == 0
)

assert np.allclose(
    phase2_df.loc[
        normal_mask,
        "temperature_c"
    ],
    phase2_df.loc[
        normal_mask,
        "original_temperature_c"
    ],
    equal_nan=True
)

assert np.allclose(
    phase2_df.loc[
        normal_mask,
        "pressure_hpa"
    ],
    phase2_df.loc[
        normal_mask,
        "original_pressure_hpa"
    ],
    equal_nan=True
)

assert np.allclose(
    phase2_df.loc[
        normal_mask,
        "relative_humidity_pct"
    ],
    phase2_df.loc[
        normal_mask,
        "original_relative_humidity_pct"
    ],
    equal_nan=True
)

print("Normal observations remain unchanged.")

Normal observations remain unchanged.


In [174]:
sensor_cols = [
    "temperature_c",
    "pressure_hpa",
    "relative_humidity_pct"
]

missing_sensor_mask = (
    phase2_df[sensor_cols]
    .isna()
    .any(axis=1)
)

dropout_mask = (
    phase2_df["anomaly_type"] == "dropout"
)

print(
    "Rows containing injected missing values:",
    int(missing_sensor_mask.sum())
)

print(
    "Dropout-labeled rows:",
    int(dropout_mask.sum())
)

assert (
    missing_sensor_mask
    == dropout_mask
).all()

print("Dropout validation passed.")

Rows containing injected missing values: 629
Dropout-labeled rows: 629
Dropout validation passed.


In [175]:
for idx in phase2_df.index[dropout_mask]:

    parameter = phase2_df.at[
        idx,
        "anomaly_parameter"
    ]

    column = sensor_columns[
        parameter
    ]

    assert pd.isna(
        phase2_df.at[
            idx,
            column
        ]
    )

print("Dropout parameter mapping passed.")

Dropout parameter mapping passed.


In [176]:
phase2_df["temperature_change"] = (
    phase2_df["temperature_c"]
    - phase2_df["original_temperature_c"]
)

phase2_df["pressure_change"] = (
    phase2_df["pressure_hpa"]
    - phase2_df["original_pressure_hpa"]
)

phase2_df["humidity_change"] = (
    phase2_df["relative_humidity_pct"]
    - phase2_df[
        "original_relative_humidity_pct"
    ]
)

In [177]:
perturbation_summary = (
    phase2_df[
        phase2_df["is_anomaly"] == 1
    ]
    .groupby("anomaly_type")[
        [
            "temperature_change",
            "pressure_change",
            "humidity_change"
        ]
    ]
    .agg(["mean", "std", "min", "max"])
)

perturbation_summary

temperature_change                        \
                                         mean        std        min   
anomaly_type                                                          
bias                                 0.498530   3.992847  -9.790013   
drift                               -0.562954   3.223107 -14.236730   
dropout                              0.000000   0.000000   0.000000   
freeze                               0.165024   1.121925  -6.070000   
multivariate_inconsistency           0.073419   5.314322 -12.331646   
noise                               -0.030954   4.232072 -24.425064   
spike                               -0.084229  10.341062 -24.664733   

                                      pressure_change                       \
                                  max            mean       std        min   
anomaly_type                                                                 
bias                         9.253104        0.472113  3.580749  -8.277431   
drift                       12.773210       -0.233452  2.722613 -10.243485   
dropout                      0.000000        0.000000  0.000000   0.000000   
freeze                       5.840000        0.035829  0.434727  -1.530000   
multivariate_inconsistency  12.298841       -0.047950  4.317942 -10.475510   
noise                       23.779472       -0.021677  3.697103 -20.035885   
spike                       24.679219        0.150882  8.572049 -20.953579   

                                      humidity_change                        \
                                  max            mean        std        min   
anomaly_type                                                                  
bias                         8.261085        0.825627   5.464606 -13.497158   
drift                       11.673690        0.536261   5.911586 -21.579063   
dropout                      0.000000        0.000000   0.000000   0.000000   
freeze                       4.360000        0.374131   2.847431 -14.310000   
multivariate_inconsistency  10.453104        0.010474   8.012404 -19.463856   
noise                       20.094716        0.146104   7.638522 -37.703926   
spike                       20.903737        0.191637  12.157321 -29.211786   

                                       
                                  max  
anomaly_type                           
bias                        14.394472  
drift                       20.947622  
dropout                      0.000000  
freeze                      21.360000  
multivariate_inconsistency  19.388358  
noise                       36.424164  
spike                       29.181852

In [178]:
sample_columns = [
    "timestamp",
    "split",
    "temperature_c",
    "original_temperature_c",
    "pressure_hpa",
    "original_pressure_hpa",
    "relative_humidity_pct",
    "original_relative_humidity_pct",
    "is_anomaly",
    "anomaly_type",
    "anomaly_parameter",
    "anomaly_magnitude"
]

phase2_df.loc[
    phase2_df["is_anomaly"] == 1,
    sample_columns
].sample(
    n=20,
    random_state=42
)

,timestamp,split,temperature_c,original_temperature_c,pressure_hpa,original_pressure_hpa,relative_humidity_pct,original_relative_humidity_pct,is_anomaly,anomaly_type,anomaly_parameter,anomaly_magnitude
279489,2014-04-25 22:10:00,train,16.170000,16.17,984.890000,985.96,68.310000,68.31,1,freeze,pressure,-1.070000
43428,2009-10-29 14:30:00,train,0.355428,11.21,995.350000,995.35,78.500000,78.50,1,spike,temperature,-10.854572
155764,2011-12-18 17:10:00,train,-12.888940,1.37,982.240000,982.24,76.300000,76.30,1,spike,temperature,-14.258940
217656,2013-02-20 12:30:00,train,-1.620000,-1.62,992.919304,995.01,79.900000,79.90,1,drift,pressure,-2.090696
128808,2011-06-14 12:30:00,train,20.920000,20.92,990.310000,990.31,50.750646,59.68,1,multivariate_inconsistency,humidity,-8.929354
175814,2012-05-05 22:50:00,train,7.130000,7.13,980.650000,980.65,102.892918,95.60,1,bias,humidity,7.292918
110907,2011-02-10 05:00:00,train,-4.610000,-4.61,995.010000,995.01,101.648332,93.50,1,drift,humidity,8.148332
353689,2015-09-23 20:50:00,validation,35.695630,11.77,988.230000,988.23,80.200000,80.20,1,spike,temperature,23.925630
6663,2009-02-16 06:40:00,train,19.059294,-1.67,991.230000,991.23,89.800000,89.80,1,spike,temperature,20.729294
155085,2011-12-14 00:00:00,train,5.510000,5.51,NaN,978.20,81.000000,81.00,1,dropout,pressure,978.200000


In [179]:
anomaly_type_counts = (
    phase2_df[
        phase2_df["is_anomaly"] == 1
    ]
    ["anomaly_type"]
    .value_counts()
    .sort_index()
)

anomaly_type_counts

anomaly_type
bias                          1889
drift                         2520
dropout                        629
freeze                        2520
multivariate_inconsistency     638
noise                         1260
spike                         3150
Name: count, dtype: int64

In [180]:
split_anomaly_counts = (
    phase2_df[
        phase2_df["is_anomaly"] == 1
    ]
    ["split"]
    .value_counts()
)

split_anomaly_counts

split
train         8824
validation    1891
test          1891
Name: count, dtype: int64

In [181]:
parameter_counts = (
    phase2_df[
        phase2_df["is_anomaly"] == 1
    ]
    ["anomaly_parameter"]
    .value_counts()
)

parameter_counts

anomaly_parameter
temperature    4421
humidity       4098
pressure       4087
Name: count, dtype: int64

In [182]:
phase2_df = phase2_df.drop(
    columns=[
        "temperature_change",
        "pressure_change",
        "humidity_change"
    ]
)

phase2_df.columns.tolist()

['timestamp',
 'station_id',
 'temperature_c',
 'pressure_hpa',
 'relative_humidity_pct',
 'original_temperature_c',
 'original_pressure_hpa',
 'original_relative_humidity_pct',
 'is_anomaly',
 'anomaly_type',
 'anomaly_parameter',
 'anomaly_magnitude',
 'split']

In [183]:
assert len(phase2_df) == 420224

assert phase2_df["timestamp"].notna().all()

assert (
    phase2_df["is_anomaly"].sum()
    == sum(split_budgets.values())
)

assert phase2_df[
    "original_temperature_c"
].notna().all()

assert phase2_df[
    "original_pressure_hpa"
].notna().all()

assert phase2_df[
    "original_relative_humidity_pct"
].notna().all()

assert (
    phase2_df["split"]
    .isin(
        [
            "train",
            "validation",
            "test"
        ]
    )
    .all()
)

print("Final Phase 2 integrity checks passed.")

Final Phase 2 integrity checks passed.


In [184]:
total_anomalies = int(
    phase2_df["is_anomaly"].sum()
)

actual_anomaly_rate = float(
    phase2_df["is_anomaly"].mean()
)

injection_report = {
    "dataset": "Jena Climate 2009-2016",

    "random_seed": RANDOM_SEED,

    "input_rows": int(len(df)),

    "canonical_rows_after_deduplication":
        int(len(phase2_df)),

    "duplicates_removed":
        int(len(df) - len(phase2_df)),

    "target_anomaly_rate_per_split":
        TARGET_ANOMALY_RATE,

    "overall_actual_anomaly_rate":
        actual_anomaly_rate,

    "total_injected_anomalies":
        total_anomalies,

    "split_summary": {},

    "anomaly_type_counts": {
        str(key): int(value)
        for key, value in
        phase2_df.loc[
            phase2_df["is_anomaly"] == 1,
            "anomaly_type"
        ]
        .value_counts()
        .items()
    },

    "anomaly_parameter_counts": {
        str(key): int(value)
        for key, value in
        phase2_df.loc[
            phase2_df["is_anomaly"] == 1,
            "anomaly_parameter"
        ]
        .value_counts()
        .items()
    },

    "robust_statistics_from_train":
        robust_stats,

    "anomaly_distribution":
        anomaly_distribution
}


for split_name in [
    "train",
    "validation",
    "test"
]:

    subset = phase2_df[
        phase2_df["split"] == split_name
    ]

    injection_report[
        "split_summary"
    ][split_name] = {

        "rows": int(len(subset)),

        "normal_rows": int(
            (subset["is_anomaly"] == 0).sum()
        ),

        "anomaly_rows": int(
            subset["is_anomaly"].sum()
        ),

        "anomaly_rate": float(
            subset["is_anomaly"].mean()
        )
    }

injection_report

{'dataset': 'Jena Climate 2009-2016',
 'random_seed': 42,
 'input_rows': 420551,
 'canonical_rows_after_deduplication': 420224,
 'duplicates_removed': 327,
 'target_anomaly_rate_per_split': 0.03,
 'overall_actual_anomaly_rate': 0.02999828662808407,
 'total_injected_anomalies': 12606,
 'split_summary': {'train': {'rows': 294156,
   'normal_rows': 285332,
   'anomaly_rows': 8824,
   'anomaly_rate': 0.02999768830144549},
  'validation': {'rows': 63034,
   'normal_rows': 61143,
   'anomaly_rows': 1891,
   'anomaly_rate': 0.029999682710917918},
  'test': {'rows': 63034,
   'normal_rows': 61143,
   'anomaly_rows': 1891,
   'anomaly_rate': 0.029999682710917918}},
 'anomaly_type_counts': {'spike': 3150,
  'drift': 2520,
  'freeze': 2520,
  'bias': 1889,
  'noise': 1260,
  'multivariate_inconsistency': 638,
  'dropout': 629},
 'anomaly_parameter_counts': {'temperature': 4421,
  'humidity': 4098,
  'pressure': 4087},
 'robust_statistics_from_train': {'temperature': {'median': 9.18,
   'q1': 2.98

In [185]:
phase2_df.to_parquet(
    OUTPUT_FILE,
    index=False
)

print(OUTPUT_FILE)

C:\Anomaly_Detection\data\processed\phase2_labeled_observations.parquet


In [186]:
with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        injection_report,
        file,
        indent=4
    )

print(REPORT_FILE)

C:\Anomaly_Detection\data\processed\anomaly_injection_report.json


In [187]:
saved_df = pd.read_parquet(
    OUTPUT_FILE
)

with open(
    REPORT_FILE,
    "r",
    encoding="utf-8"
) as file:

    saved_report = json.load(file)

print("Saved rows:", len(saved_df))
print(
    "Saved anomalies:",
    int(saved_df["is_anomaly"].sum())
)

Saved rows: 420224
Saved anomalies: 12606


In [188]:
print(
    f"Canonical Rows               : "
    f"{len(phase2_df):,}"
)

print(
    f"Duplicates Removed           : "
    f"{len(df) - len(phase2_df):,}"
)

print(
    f"Injected Anomalies           : "
    f"{total_anomalies:,}"
)

print(
    f"Overall Anomaly Rate         : "
    f"{actual_anomaly_rate:.4%}"
)

print()

for split_name in [
    "train",
    "validation",
    "test"
]:

    subset = phase2_df[
        phase2_df["split"] == split_name
    ]

    print(
        f"{split_name.title():<12}"
        f"Rows : {len(subset):>7,}   "
        f"Anomalies : "
        f"{int(subset['is_anomaly'].sum()):>5,}   "
        f"Rate : "
        f"{subset['is_anomaly'].mean():.4%}"
    )

print()

for anomaly_type, count in (
    phase2_df.loc[
        phase2_df["is_anomaly"] == 1,
        "anomaly_type"
    ]
    .value_counts()
    .items()
):

    print(
        f"{anomaly_type:<30}: "
        f"{count:,}"
    )

print()

for parameter, count in (
    phase2_df.loc[
        phase2_df["is_anomaly"] == 1,
        "anomaly_parameter"
    ]
    .value_counts()
    .items()
):

    print(
        f"{parameter:<30}: "
        f"{count:,}"
    )

Canonical Rows               : 420,224
Duplicates Removed           : 327
Injected Anomalies           : 12,606
Overall Anomaly Rate         : 2.9998%

Train       Rows : 294,156   Anomalies : 8,824   Rate : 2.9998%
Validation  Rows :  63,034   Anomalies : 1,891   Rate : 3.0000%
Test        Rows :  63,034   Anomalies : 1,891   Rate : 3.0000%

spike                         : 3,150
drift                         : 2,520
freeze                        : 2,520
bias                          : 1,889
noise                         : 1,260
multivariate_inconsistency    : 638
dropout                       : 629

temperature                   : 4,421
humidity                      : 4,098
pressure                      : 4,087
